In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchinfo import summary
from torchviz import make_dot
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torchvision.datasets as datasets

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
print(device)

cuda


In [18]:
# 1. 変換ルールの定義
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 2. データセットの作成（transformを適用）
train_dataset = datasets.CIFAR10(root='./data', train=True, download=False, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=False, transform=transform)

# 3. 🌟 データローダーの作成（これがラベルをTensorにまとめてくれます！）
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

# --- 確認用コード ---
# 試しに1バッチだけ取り出してみて、型を確認します
images, labels = next(iter(train_loader))
print("imagesの型:", type(images), " / 形:", images.shape)
print("labelsの型:", type(labels), " / 形:", labels.shape)

imagesの型: <class 'torch.Tensor'>  / 形: torch.Size([64, 3, 32, 32])
labelsの型: <class 'torch.Tensor'>  / 形: torch.Size([64])


In [19]:
class CIFAR10CNN(nn.Module):
    def __init__(self):
        super(CIFAR10CNN, self).__init__()
        
        # ==========================================
        # 1. 畳み込み層（Convolutional Layers）の特徴抽出部分
        # ==========================================
        self.features = nn.Sequential(
            # 1層目: 入力3チャンネル(RGB) -> 出力32フィルター
            # padding=1 を指定し、畳み込みによって画像サイズが縮むのを防ぎます
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 画像サイズを半分に (32x32 -> 16x16)

            # 2層目: 入力32 -> 出力64フィルター
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # (16x16 -> 8x8)

            # 3層目: 入力64 -> 出力128フィルター
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  # (8x8 -> 4x4)
        )
        
        # 1次元に平坦化するレイヤー
        self.flatten = nn.Flatten()
        
        # ==========================================
        # 2. 全結合層（Fully-connected Layers）の分類部分
        # ==========================================
        # 図の指示通り、活性化関数には ReLU ではなく Tanh を使用します
        self.classifier = nn.Sequential(
            # 入力サイズ: 128(フィルター数) * 4(縦) * 4(横) = 2048
            nn.Linear(2048, 128),
            nn.Tanh(),
            
            nn.Linear(128, 32),
            nn.Tanh(),
            
            nn.Linear(32, 5),
            nn.Tanh(),
            
            nn.Linear(5, 5),
            nn.Tanh(),
            
            # 最終出力: 10クラスの one-hot vector 予測用
            # PyTorchのCrossEntropyLossを使う場合、最終層には活性化関数をかけません
            nn.Linear(5, 10)
        )

    def forward(self, x):
        # 1. 画像から特徴を抽出
        x = self.features(x)
        
        # 2. 2次元の画像特徴を1次元のベクトル(長さ2048)に平坦化
        x = self.flatten(x)
        
        # 3. 全結合層で最終的な10クラスの確率（ロジット）を計算
        x = self.classifier(x)
        
        return x

model = CIFAR10CNN().to(device)

In [26]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [27]:
epochs = 500

In [28]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        # ★データをGPUへ転送
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

Epoch [1/500], Loss: 0.4175
Epoch [2/500], Loss: 0.3508
Epoch [3/500], Loss: 0.3233
Epoch [4/500], Loss: 0.2969
Epoch [5/500], Loss: 0.2834
Epoch [6/500], Loss: 0.2699
Epoch [7/500], Loss: 0.2611
Epoch [8/500], Loss: 0.2511
Epoch [9/500], Loss: 0.2419
Epoch [10/500], Loss: 0.2343
Epoch [11/500], Loss: 0.2280
Epoch [12/500], Loss: 0.2223
Epoch [13/500], Loss: 0.2171
Epoch [14/500], Loss: 0.2165
Epoch [15/500], Loss: 0.2095
Epoch [16/500], Loss: 0.2021
Epoch [17/500], Loss: 0.1950
Epoch [18/500], Loss: 0.1899
Epoch [19/500], Loss: 0.1887
Epoch [20/500], Loss: 0.1823
Epoch [21/500], Loss: 0.1779
Epoch [22/500], Loss: 0.1737
Epoch [23/500], Loss: 0.1723
Epoch [24/500], Loss: 0.1671
Epoch [25/500], Loss: 0.1652
Epoch [26/500], Loss: 0.1598
Epoch [27/500], Loss: 0.1542
Epoch [28/500], Loss: 0.1512
Epoch [29/500], Loss: 0.1504
Epoch [30/500], Loss: 0.1441
Epoch [31/500], Loss: 0.1407
Epoch [32/500], Loss: 0.1389
Epoch [33/500], Loss: 0.1346
Epoch [34/500], Loss: 0.1319
Epoch [35/500], Loss: 0

In [29]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        # ★評価時もデータをGPUへ転送
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"\nテストデータでの正解率: {100 * correct / total:.2f}%")


テストデータでの正解率: 73.11%
